# Ch01-07: 피처 엔지니어링과 도메인 지식 — 모범답안


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
np.random.seed(42)

---
## 문제 1 풀이

상호작용 항 AIC/BIC 비교

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression
np.random.seed(42); n=500
x1,x2,x3 = np.random.uniform(1,10,n), np.random.uniform(1,10,n), np.random.uniform(1,10,n)
y = 3+2*x1+1.5*x2+0.5*x1*x2+np.random.normal(0,2,n)

models = {
    'x1+x2': np.column_stack([x1,x2]),
    'x1+x2+x1x2': np.column_stack([x1,x2,x1*x2]),
    'x1+x2+x1x3': np.column_stack([x1,x2,x1*x3]),
    'x1+x2+x1x2+x2x3': np.column_stack([x1,x2,x1*x2,x2*x3]),
}
for name, X in models.items():
    lr = LinearRegression().fit(X, y); yp = lr.predict(X)
    rss = np.sum((y-yp)**2); k=X.shape[1]+1
    aic = n*np.log(rss/n)+2*k; bic = n*np.log(rss/n)+k*np.log(n)
    print(f"{name:25s}: R²={lr.score(X,y):.4f}, AIC={aic:.1f}, BIC={bic:.1f}")


**결과 해석**: 진짜 상호작용(x1*x2)을 포함한 모형이 AIC/BIC 모두 최소. 가짜 상호작용(x1*x3)은 개선 없이 복잡도만 증가.

---
## 문제 2 풀이

금융 도메인 피처 설계

In [ ]:
import numpy as np, pandas as pd
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_val_score

np.random.seed(42); n=1000
prices = np.cumsum(np.random.normal(0.001,0.02,n))+100
volumes = np.random.lognormal(10,0.5,n)
y = (np.diff(np.append(prices,prices[-1]+np.random.normal()))>0).astype(int)[:n]
df = pd.DataFrame({'price':prices,'volume':volumes,'y':y})

# 원시 피처
X_raw = df[['price','volume']].values

# 도메인 피처
df['ma5'] = df['price'].rolling(5,min_periods=1).mean()
df['ma20'] = df['price'].rolling(20,min_periods=1).mean()
df['ma_ratio'] = df['ma5']/df['ma20']
df['volatility'] = df['price'].rolling(10,min_periods=1).std()
df['momentum'] = df['price'].pct_change(5)
df['vol_ma'] = df['volume'].rolling(5,min_periods=1).mean()
df['vol_ratio'] = df['volume']/(df['vol_ma']+1)

# RSI-like
delta = df['price'].diff()
gain = delta.clip(lower=0).rolling(14,min_periods=1).mean()
loss = (-delta.clip(upper=0)).rolling(14,min_periods=1).mean()
df['rsi'] = 100 - 100/(1+gain/(loss+1e-10))
df = df.fillna(0)

feat_cols = ['ma_ratio','volatility','momentum','vol_ratio','rsi']
X_domain = df[feat_cols].values

raw_cv = cross_val_score(GradientBoostingClassifier(n_estimators=50,max_depth=3,random_state=42), X_raw, y, cv=5).mean()
dom_cv = cross_val_score(GradientBoostingClassifier(n_estimators=50,max_depth=3,random_state=42), X_domain, y, cv=5).mean()
print(f"Raw features CV: {raw_cv:.3f}")
print(f"Domain features CV: {dom_cv:.3f}")


**결과 해석**: 도메인 지식 기반 피처(이동평균 비율, RSI 등)가 원시 피처(가격, 거래량)보다 우수한 예측 성능 제공.

---
## 문제 3 풀이

자동 피처 생성 + 상호정보량 선택

In [ ]:
import numpy as np
from sklearn.feature_selection import mutual_info_regression
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score
from itertools import combinations

np.random.seed(42); n=500; p=5
X = np.random.randn(n,p)
y = 2*X[:,0]*X[:,1] + X[:,2]**2 - 3*X[:,3] + np.random.normal(0,0.5,n)

def auto_feature_gen(X, y, k=20):
    n,p = X.shape; new_feats = []; names = []
    for i,j in combinations(range(p),2):
        new_feats.append(X[:,i]*X[:,j]); names.append(f'x{i}*x{j}')
        new_feats.append(X[:,i]+X[:,j]); names.append(f'x{i}+x{j}')
        denom = X[:,j].copy(); denom[np.abs(denom)<0.01]=0.01
        new_feats.append(X[:,i]/denom); names.append(f'x{i}/x{j}')
    for i in range(p):
        new_feats.append(X[:,i]**2); names.append(f'x{i}^2')
    X_new = np.column_stack(new_feats)
    mi = mutual_info_regression(X_new, y, random_state=42)
    top_k = np.argsort(mi)[-k:]
    return X_new[:, top_k], [names[i] for i in top_k], mi[top_k]

X_auto, feat_names, mis = auto_feature_gen(X, y, k=10)
print("상위 피처 (상호정보량):")
for name, mi_val in sorted(zip(feat_names, mis), key=lambda x:-x[1]):
    print(f"  {name}: MI={mi_val:.4f}")

base_cv = cross_val_score(LinearRegression(), X, y, cv=5, scoring='r2').mean()
aug_cv = cross_val_score(LinearRegression(), np.column_stack([X, X_auto]), y, cv=5, scoring='r2').mean()
print(f"\n원본 R²: {base_cv:.4f}, 확장 R²: {aug_cv:.4f}")


**결과 해석**: 상호정보량으로 선택된 피처(x0*x1, x2^2)가 실제 데이터 생성 메커니즘과 일치. 자동 생성+선택으로 R² 향상.

---
## 문제 4 풀이

다항식 차수 + 정규화 편향-분산

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import cross_val_score

np.random.seed(42); n=200
x = np.random.uniform(0,5,n); y = np.sin(x)+0.3*np.random.randn(n)
X = x.reshape(-1,1)

degrees = range(1,16)
results = {'OLS':[],'Ridge':[],'Lasso':[]}
for d in degrees:
    Xp = PolynomialFeatures(d,include_bias=False).fit_transform(X)
    for name, model in [('OLS',LinearRegression()),('Ridge',Ridge(alpha=1)),('Lasso',Lasso(alpha=0.01))]:
        cv = cross_val_score(model, Xp, y, cv=5, scoring='neg_mean_squared_error').mean()
        results[name].append(-cv)

fig,ax = plt.subplots(figsize=(10,5))
for name in results: ax.plot(list(degrees), results[name], 'o-', label=name)
ax.set_xlabel('다항식 차수'); ax.set_ylabel('CV MSE'); ax.legend()
ax.set_title('차수에 따른 정규화 효과'); plt.tight_layout(); plt.show()


**결과 해석**: OLS는 고차에서 과적합(MSE 급증). Ridge/Lasso는 정규화로 고차 계수를 억제하여 과적합 완화. 최적 차수는 편향-분산 균형점.

---
## 확장 토론

### 피처 엔지니어링 원칙

1. 도메인 지식 우선 (물리적 의미 있는 변환)
2. 상호작용은 AIC/BIC로 검증
3. 자동 생성은 선택과 결합
4. 정규화로 과적합 방지